# Imports

In [1]:
import cda2
#import datetime
import pyspark.sql.functions as F
import pyspark.sql.types as T
import json

from datetime import datetime, timedelta
from pyspark.sql.window import Window
from pyspark.sql.functions import col, row_number

# Connect to Spark

In [2]:
api = cda2.Api()

Set configuration parameters to better optimize queries.

In [3]:
config = {
    "spark.sql.adaptive.enabled": "true",
    "spark.sql.adaptive.coalescePartitions.enabled": "true",
    "spark.sql.adaptive.coalescePartitions.parallelismFirst": "false",
    "spark.sql.adaptive.coalescePartitions.minPartitionSize": "1m",
    "spark.executor.memory": "8g",
    "spark.executor.memoryOverhead": "16g",
}

Start Spark and specify number of cpus to use.

In [4]:
#api.start_spark(n_executors=100, config=config)
api.start_spark(n_executors=400)

https://artifacts.mitre.org/artifactory/java-libs-release added as a remote repository with the name: repo-1
https://dali.mitre.org/nexus/content/repositories/mitre-caasd-releases added as a remote repository with the name: repo-2
https://dali.mitre.org/nexus/content/repositories/external-releases added as a remote repository with the name: repo-3


:: loading settings :: url = jar:file:/devel/data_access/software/tdp-jupyter/poetry/cache/virtualenvs/python39-QwwvzYkJ-py3.9/lib/python3.9/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/rchong/.ivy2/cache
The jars for the packages stored in: /home/rchong/.ivy2/jars
org.mitre.spark#spark-geo_spark3.5_2.12 added as a dependency
org.apache.spark#spark-avro_2.12 added as a dependency
graphframes#graphframes added as a dependency
org.mongodb.spark#mongo-spark-connector_2.12 added as a dependency
com.oracle.database.jdbc#ojdbc8 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-6003b317-2fc4-4387-87b2-657d4d4c0c43;1.0
	confs: [default]
	found org.mitre.spark#spark-geo_spark3.5_2.12;0.2.0 in repo-1
	found org.mitre.spark#spark-geo-core_2.12;0.2.0 in repo-1
	found org.scala-lang.modules#scala-collection-compat_2.12;2.11.0 in central
	found net.sf.geographiclib#GeographicLib-Java;2.0 in central
	found org.ejml#ejml-core;0.43.1 in central
	found org.ejml#ejml-ddense;0.43.1 in central
	found com.esri.geometry#esri-geometry-api;2.2.4 in central
	found com.fasterxml.jackson.core#jackson-core;2.9.6 in central
	fou

# Define global variables and functions

In [5]:
airports = [
    "KIAD",
]

In [6]:
%run ./shared_variables.ipynb

Function to convert Unix timestamp (milliseconds from 1970) to YYYYMMDD string.

In [7]:
#year0 = "2025"
year1 = str(int(year0) + 1)

In [8]:
dates = {"start_date": year0 + "-01-01", "end_date": year1 +"-01-01"}
print(dates)

{'start_date': '2025-01-01', 'end_date': '2026-01-01'}


In [9]:
print("retrieving procedures: ", datetime.now())

retrieving procedures:  2026-04-02 08:00:19.880155


# Retrieve SIDs and STARs for the airports of interest

<li>only get SID and STAR procedure types. (procedure_type = "APPROACH" is ignored)</li>
<li>create a unique <i>grouping</i> column.</li>
</br>
this will retrieve multiple versions of each fix, one for each update cycle in the year.

In [34]:
# v0 of this notebook didn't work when i tried it in oct 2025. this slack chat with matt pollock helped me figure out the issues...
# https://mitre.slack.com/archives/C2C46R03F/p1759873151426549

df_procedures = (
    api.dataframe("ArincTransition", **dates, metadata=True)
#    .filter(F.col("navigation_source") == "CIFP")
    .filter(F.col("procedure_name") == "GIBBZ5")
#    .filter(F.col("transition_name") == "ALL")
    .filter(F.col("metadata.effective_end_date") == 1769040000000)
)

Multiple versions found: 3.1.71, 3.1.73, 3.1.74, 3.1.75, 3.1.76, 3.1.77, 3.1.79, 3.1.80
                                                                                

In [10]:
# v0 of this notebook didn't work when i tried it in oct 2025. this slack chat with matt pollock helped me figure out the issues...
# https://mitre.slack.com/archives/C2C46R03F/p1759873151426549

df_procedures = (
    api.dataframe("ArincTransition", **dates, metadata=True)
    .select(
        F.col("primary_key").alias("segment_id"),
        "dtpp_procedure_name",
        "procedure_name",
        "transition_name",
        "procedure_type",
        "transition_type",
        "navigation_source",
        F.col("airport.icao_region").alias("airport_icao_region"),
#        F.col("airport.name").alias("airport"),
        F.explode("legs").alias("leg"),
        F.col("metadata.effective_end_date").alias("end_date")
    )
    .withColumn("fix_name", F.col("leg.path_terminator.identification.name"))
    .withColumn("fix_icao_region", F.col("leg.path_terminator.identification.icao_region"))
    .withColumn("seq", F.col("leg.sequence_number").alias("seq"))
    .withColumn("latitude", F.col("leg.path_terminator.latitude"))
    .withColumn("longitude", F.col("leg.path_terminator.longitude"))
    .withColumn("magnetic_variation", F.col("leg.path_terminator.magnetic_variation.modeled"))
    .withColumn("speed_description", F.col("leg.speed_limit.descriptor"))
    .withColumn("speed_limit", F.col("leg.speed_limit.limit"))
    .withColumn("speed_altitude", F.col("leg.speed_limit.altitude"))
    .withColumn("altitude_description", F.col("leg.altitude_limits.description"))
    .withColumn("altitude_value1", F.col("leg.altitude_limits.value1"))
    .withColumn("altitude_value2", F.col("leg.altitude_limits.value2"))
    .withColumn("transition_name", F.coalesce(F.col("transition_name"), F.lit("n/a")))
#    .withColumn("grouping", F.concat("procedure_name", F.lit("_"), "transition_name", F.lit("_"), F.format_string("%03d", F.col("seq")), F.lit("_"), "fix_name"))
    .withColumn("grouping", F.concat("navigation_source", F.lit("_"), "procedure_name", F.lit("_"), "transition_name", F.lit("_"), F.format_string("%03d", F.col("seq")), F.lit("_"), "fix_name"))
    .drop("leg")
    .dropDuplicates(["path_terminators"])
    .orderBy("grouping", col("end_date").desc())
#    .persist()
)


df_procedures = (
    df_procedures
    .filter(F.col("fix_name").isNotNull())
    .filter(F.col("latitude").isNotNull())
    .filter(F.col("longitude").isNotNull())
    .filter(F.col("magnetic_variation").isNotNull())
    .filter(F.col("procedure_type").isin("SID", "STAR"))
    .filter(F.col("airport_icao_region").startswith("K") | F.col("airport_icao_region").startswith("PA") | F.col("airport_icao_region").startswith("PH"))

#    .filter(F.col("navigation_source").isin(["CIFP", "LIDO"]))
#    .filter(F.col("end_date") == 1769040000000)
#    .filter(F.col("navigation_source") == "CIFP")
#    .filter(F.col("navigation_source") == "LIDO")

#    .filter(F.col("procedure_name") == "LGTNG5")
#    .filter(F.col("procedure_name") == "BNFSH2")
#    .filter(F.col("procedure_name") == "ATL2")
#    .filter(F.col("procedure_name") == "SCOT7")
#    .filter(F.col("procedure_name") == "CAVLR6")
    .filter(F.col("procedure_name") == "GIBBZ5")
#    .filter(F.col("procedure_name") == "SUNSS8")
#    .filter(F.col("procedure_name").isin(["GIBBZ5", "CAVLR6"]))

#    .filter(F.col("fix_name") == "MEEGO")

#    comment the following out so we get procedures for all US airports
#    .filter(F.col("airport").isin(airports))

#    .filter(F.col("grouping").isNotNull())
#    .filter(F.col("transition_name").isNotNull())
#    .filter(F.col("fix_name").isNotNull())
#    .persist()
#    .distinct()
)

Multiple versions found: 3.1.71, 3.1.73, 3.1.74, 3.1.75, 3.1.76, 3.1.77, 3.1.79, 3.1.80
                                                                                

In [35]:
df_procedures.count()

14

In [46]:
#df_procedures.select("path_terminators").show(truncate=False)
#df_procedures.dropDuplicates(["path_terminators"]).orderBy("path_terminators").show()

In [ ]:
#(
#    df_procedures
#    .orderBy("transition_name")
#    .show(df_procedures.count())
#)

In [ ]:
# get the newest entry based on the end_date

window = Window.partitionBy("grouping").orderBy(col("end_date").desc())

df_procedures_newest = (df_procedures
    .withColumn("row", row_number().over(window))
    .filter(col("row") == 1)
    .drop("row", "end_date")
    .orderBy("grouping")
)

In [ ]:
df_procedures_newest.count()

In [ ]:
#df_procedures_newest.show(df_procedures_newest.count())

In [ ]:
# hack to just use one navigation_source. if there's two, use the first; don't care which; if there's one, then just use it.

df_available_nav_sources = (
    df_procedures_newest
    .select(
        "procedure_name",
        "navigation_source",
    )
    .distinct()
    .orderBy("procedure_name")
)

#df_available_nav_sources.count()

#df_available_nav_sources.first()["navigation_source"]

#df_procedures_newest = (
#    df_procedures_newest
#    .filter(col("navigation_source") == df_available_nav_sources.first()["navigation_source"])
#)

In [ ]:
#df_available_nav_sources.show()

In [ ]:
# choose only one navigation_source

#window = Window.partitionBy("procedure_name").orderBy(col("navigation_source").desc())
window = Window.partitionBy("procedure_name").orderBy(col("navigation_source"))
df_available_nav_sources = (df_available_nav_sources
    .withColumn("row", row_number().over(window))
    .filter(col("row") == 1)
    .drop("row")
)

#df_available_nav_sources.count()
#df_available_nav_sources.show()

In [ ]:
df_procedures_newest = df_procedures_newest.join(df_available_nav_sources, on=["procedure_name", "navigation_source"], how="left_semi")
df_procedures_newest.count()

In [ ]:
# df_procedures_newest.show(df_procedures_newest.count())
# df_procedures_newest.show()

In [ ]:
#window = Window.partitionBy("procedure_name").orderBy(col("navigation_source"))
#df_available_nav_sources = (df_available_nav_sources
#    .withColumn("row", row_number().over(window))
#    .filter(col("row") == 1)
#    .drop("row")
#)

In [ ]:
# df_available_nav_sources.count()

In [ ]:
#df_available_nav_sources.show(100)

In [ ]:
(
    df_procedures_newest
        .select(
            "dtpp_procedure_name",
            "navigation_source",
            "procedure_name",
            "procedure_type",
            "transition_name",
            "transition_type",
            "fix_name",
            "fix_icao_region",
            "seq",
            "latitude",
            "longitude",
            "magnetic_variation",
#            "airport",
            "speed_description",
            "speed_limit",
            "speed_altitude",
            "altitude_description",
            "altitude_value1",
            "altitude_value2",
        )
#        .coalesce(1)
        .withColumn("proc_type", F.col("procedure_type"))
        .withColumn("proc_name", F.col("procedure_name"))
        .repartition("proc_type", "proc_name")
        .write.option("header", True)
        .partitionBy(["proc_type", "proc_name"])
        .csv("CRAFT/" + year0 + "/procedures", compression="None", mode="overwrite")
)

## output the procedures (SIDs and STARs) for each airport

In [ ]:
# v0 of this notebook didn't work when i tried it in oct 2025. this slack chat with matt pollock helped me figure out the issues...
# https://mitre.slack.com/archives/C2C46R03F/p1759873151426549

df_procedures_by_airports = (
    api.dataframe("ArincTransition", **dates, metadata=True)
    .select(
        "dtpp_procedure_name",
        "procedure_name",
        "procedure_type",
        "navigation_source",
        F.col("airport.icao_region").alias("airport_icao_region"),
        F.col("airport.name").alias("airport"),
        F.col("metadata.effective_end_date").alias("end_date")
    )
    .withColumn("grouping", F.concat("airport", F.lit("_"), "procedure_name"))
    .filter(F.col("airport_icao_region").startswith("K") | F.col("airport_icao_region").startswith("PA") | F.col("airport_icao_region").startswith("PH"))
#    .filter(F.col("navigation_source") == "CIFP")
    .filter(F.col("procedure_type").isin("SID", "STAR"))

#    comment the following out so we get procedures for all US airports
#    .filter(F.col("airport").isin(airports)) 
#    .filter(F.col("airport") == "KIAD")

    .orderBy("grouping")
    .persist()
)

In [ ]:
window = Window.partitionBy("grouping").orderBy(col("end_date").desc())

df_procedures_by_airports_newest = (df_procedures_by_airports
    .withColumn("row", row_number().over(window))
    .filter(col("row") == 1)
    .drop("row", "end_date")
    .orderBy("procedure_type", "dtpp_procedure_name", "grouping")
)

In [ ]:
df_procedures_by_airports_newest.count()

In [ ]:
#df_procedures_by_airports_newest.show(df_procedures_by_airports_newest.count())
#df_procedures_by_airports_newest.show()

In [ ]:
(
    df_procedures_by_airports_newest
        .select(
            "airport",
            "procedure_type",
            "procedure_name",
        )
        .orderBy("airport", "procedure_type", "procedure_name")
        .write.option("header", True)
        .csv("CRAFT/" + year0 + "/procedures/airports", compression="None", mode="overwrite")
)

## output all procedure fixes
<li>add a column that concats all the procedures that use the fix</li>
<li>add a column that concats the procedure types that use the fix</li>

In [ ]:
(
    df_procedures_newest.groupBy("fix_name", "fix_icao_region", "latitude", "longitude", "magnetic_variation")
#    df_procedures_newest
#        .select(
#            "procedure_name",
#            "procedure_type",
#            "fix_name",
#            "fix_icao_region",
#            "latitude",
#            "longitude",
#            "magnetic_variation",
#        )
#        .groupBy("fix_name", "fix_icao_region")
        .agg(F.concat_ws(":", F.collect_set("procedure_name")).alias("procedures_using_fix"), F.concat_ws(":", F.collect_set("procedure_type")).alias("procedure_types_using_fix"))
        .orderBy("fix_name")
        .write.option("header", True)
        .csv("CRAFT/" + year0 + "/procedures/all_fixes", compression="None", mode="overwrite")
)